In [6]:
# ============================================================
# RAG STEP 3: RETRIEVING RELEVANT CONTEXT
# Complete Google Colab - Single Cell
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL REQUIRED LIBRARIES
# ------------------------------------------------------------
!pip install -q langchain-text-splitters chromadb sentence-transformers

# ------------------------------------------------------------
# 2. IMPORT REQUIRED LIBRARIES
# ------------------------------------------------------------
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

# ------------------------------------------------------------
# 3. CREATE TWO SAMPLE DOCUMENTS
# ------------------------------------------------------------
documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
""",
    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
"""
}

# ------------------------------------------------------------
# 4. CHUNK DOCUMENTS
# ------------------------------------------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# ------------------------------------------------------------
# 5. CREATE CHROMADB CLIENT AND EMBEDDING FUNCTION
# ------------------------------------------------------------
client = chromadb.Client()

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# ------------------------------------------------------------
# 6. CREATE OR GET CHROMADB COLLECTION
# ------------------------------------------------------------
collection = client.get_or_create_collection(
    name="rag_chunks",
    embedding_function=hf_embedder
)

# ------------------------------------------------------------
# 7. STORE CHUNKS FROM BOTH DOCUMENTS
# ------------------------------------------------------------
for source, text in documents.items():
    doc_chunks = splitter.split_text(text)
    collection.add(
        documents=doc_chunks,
        metadatas=[
            {
                "source": source,
                "chunk_index": i
            }
            for i in range(len(doc_chunks))
        ],
        ids=[
            f"{source}_chunk_{i}"
            for i in range(len(doc_chunks))
        ]
    )

print("=" * 70)
print("DOCUMENT STORAGE COMPLETED")
print("=" * 70)
print("Total chunks stored:", collection.count())

# ------------------------------------------------------------
# 8. BASIC RETRIEVAL
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("1. BASIC RETRIEVAL")
print("=" * 70)

user_query = "How does RAG improve answers from a language model?"

results = collection.query(
    query_texts=[user_query],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Distance: {distance:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk #{meta['chunk_index']}"
    )
    print(doc)
    print("-" * 60)

# ------------------------------------------------------------
# 9. DISTANCE TO SIMILARITY
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("2. DISTANCE VS SIMILARITY")
print("=" * 70)

def distance_to_similarity(distance):
    return 1 - distance

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    similarity = distance_to_similarity(distance)
    print(
        f"Similarity: {similarity:.4f} | "
        f"Source: {meta['source']} | "
        f"Chunk #{meta['chunk_index']}"
    )

# ------------------------------------------------------------
# 10. RETRIEVAL WITH SIMILARITY THRESHOLD
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("3. SIMILARITY THRESHOLD")
print("=" * 70)

def retrieve_with_threshold(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.3
):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    filtered = []
    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = distance_to_similarity(distance)
        if similarity >= similarity_threshold:
            filtered.append({
                "text": doc,
                "metadata": meta,
                "similarity": similarity
            })
    return filtered

# ------------------------------------------------------------
# 11. TEST RELEVANT QUERY
# ------------------------------------------------------------
relevant_matches = retrieve_with_threshold(
    "What is deep learning?",
    collection,
    similarity_threshold=0.3
)

print(
    f"Relevant query -> "
    f"{len(relevant_matches)} chunk(s) passed the threshold\n"
)

for match in relevant_matches:
    print(
        f"Similarity: {match['similarity']:.4f} | "
        f"{match['text'][:100]}..."
    )

# ------------------------------------------------------------
# 12. TEST UNRELATED QUERY
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("4. UNRELATED QUERY")
print("=" * 70)

unrelated_matches = retrieve_with_threshold(
    "What is the best recipe for chocolate cake?",
    collection,
    similarity_threshold=0.3
)

print(
    f"Unrelated query -> "
    f"{len(unrelated_matches)} chunk(s) passed the threshold"
)

if len(unrelated_matches) == 0:
    print(
        "No relevant context found - "
        "a real RAG system should say "
        "'I don't know' here instead of guessing."
    )

# ------------------------------------------------------------
# 13. SEMANTIC SEARCH + METADATA FILTER
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("5. METADATA FILTERED RETRIEVAL")
print("=" * 70)

results = collection.query(
    query_texts=[
        "how does search work with meaning instead of keywords"
    ],
    n_results=2,
    where={
        "source": "embeddings_in_practice.txt"
    },
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(
        f"Similarity: "
        f"{distance_to_similarity(distance):.4f} | "
        f"Source: {meta['source']}"
    )
    print(doc)
    print("-" * 60)

# ------------------------------------------------------------
# 14. REUSABLE RETRIEVE_CONTEXT FUNCTION
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("6. REUSABLE RETRIEVE_CONTEXT FUNCTION")
print("=" * 70)

def retrieve_context(
    query,
    collection,
    n_results=5,
    similarity_threshold=0.25,
    source_filter=None
):
    query_kwargs = {
        "query_texts": [query],
        "n_results": n_results,
        "include": [
            "documents",
            "metadatas",
            "distances"
        ]
    }

    if source_filter:
        query_kwargs["where"] = {
            "source": source_filter
        }

    results = collection.query(**query_kwargs)
    context_chunks = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = distance_to_similarity(distance)
        if similarity >= similarity_threshold:
            context_chunks.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })
    return context_chunks

# Test retrieve_context()
context = retrieve_context(
    "What is machine learning?",
    collection
)

for c in context:
    print(c)

# ------------------------------------------------------------
# 15. FORMAT RETRIEVED CONTEXT FOR LLM
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("7. LLM-READY PROMPT")
print("=" * 70)

def format_context_for_llm(
    query,
    context_chunks
):
    if not context_chunks:
        return (
            f"Question: {query}\n\n"
            "No relevant context was found in the knowledge base."
        )

    context_text = "\n\n".join(
        f"[Source: {c['source']}, "
        f"chunk {c['chunk_index']}]\n"
        f"{c['text']}"
        for c in context_chunks
    )

    prompt = f"""Answer the question using ONLY the context below.
If the answer isn't in the context, say you don't know.

Context:

{context_text}

Question: {query}

Answer:"""
    return prompt

# ------------------------------------------------------------
# 16. GENERATE FINAL PROMPT
# ------------------------------------------------------------
query = "How does RAG combine search with a language model?"
context = retrieve_context(
    query,
    collection
)

final_prompt = format_context_for_llm(
    query,
    context
)

print(final_prompt)

# ------------------------------------------------------------
# 17. TEST PARAPHRASED QUESTIONS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("8. PARAPHRASED QUERY TESTING")
print("=" * 70)

test_queries = [
    "Why do companies use vector databases?",
    "What's the difference between AI and machine learning?",
    "How do chatbots understand differently worded questions?"
]

for q in test_queries:
    print("\nQuery:", q)
    matches = retrieve_context(
        q,
        collection,
        n_results=2
    )
    if not matches:
        print("   No relevant context found.")
    else:
        for m in matches:
            print(
                f"   [{m['similarity']}] "
                f"({m['source']}) "
                f"{m['text'][:100]}..."
            )

# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("RAG STEP 3 COMPLETED SUCCESSFULLY")
print("=" * 70)
print("Total chunks stored:", collection.count())
print("Basic retrieval: Completed")
print("Distance calculation: Completed")
print("Similarity conversion: Completed")
print("Similarity threshold: Completed")
print("Metadata filtering: Completed")
print("Reusable retrieve_context(): Completed")
print("LLM prompt formatting: Completed")
print("Paraphrased query testing: Completed")
print("=" * 70)
print("READY FOR RAG STEP 4: LLM GENERATION")
print("=" * 70)

DOCUMENT STORAGE COMPLETED
Total chunks stored: 25

1. BASIC RETRIEVAL
Distance: 0.3244 | Source: ai_overview.txt | Chunk #6
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
------------------------------------------------------------
Distance: 0.3550 | Source: ai_overview.txt | Chunk #6
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
------------------------------------------------------------
Distance: 0.6356 | Source: ai_overview.txt | Chunk #7
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
---